In [ ]:
import pandas as pd

colunas_obrigatorias = {
    "Data_Movimentacao__c",
    "Tipo_Movimentacao__c",
    "Produto__c",
    "Quantidade__c",
}
colunas_ausentes = colunas_obrigatorias.difference(df_combinado.columns)
if colunas_ausentes:
    raise ValueError(f"Colunas ausentes em df_combinado: {sorted(colunas_ausentes)}")

df_combinado = df_combinado.copy()
df_combinado["Data_Movimentacao__c"] = pd.to_datetime(
    df_combinado["Data_Movimentacao__c"], errors="coerce"
)
df_combinado = df_combinado.dropna(subset=["Data_Movimentacao__c"])

df_combinado["Ano"] = df_combinado["Data_Movimentacao__c"].dt.year
df_combinado["Mes"] = df_combinado["Data_Movimentacao__c"].dt.month
df_combinado["Dia_Semana"] = df_combinado["Data_Movimentacao__c"].dt.dayofweek
df_combinado["Dia_Mes"] = df_combinado["Data_Movimentacao__c"].dt.day
df_combinado["Semana_Ano"] = (
    df_combinado["Data_Movimentacao__c"].dt.isocalendar().week.astype("int64")
)
df_combinado["Eh_Fim_Semana"] = df_combinado["Dia_Semana"] >= 5

tipo_movimentacao = (
    df_combinado["Tipo_Movimentacao__c"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.casefold()
    .replace({"saída": "saida"})
)
df_saidas = df_combinado.loc[tipo_movimentacao.eq("saida")].copy()

grupo = ["Produto__c", "Ano", "Mes"]
if "Categoria__c_produto" in df_saidas.columns:
    grupo.append("Categoria__c_produto")

df_demanda_mensal = (
    df_saidas.groupby(grupo, as_index=False)["Quantidade__c"]
    .sum()
    .rename(columns={"Quantidade__c": "Demanda_Mensal"})
)

if "Categoria__c_produto" in df_demanda_mensal.columns:
    df_demanda_mensal = pd.get_dummies(
        df_demanda_mensal, columns=["Categoria__c_produto"], dtype=int
    )

print(df_demanda_mensal)
df_demanda_mensal.info()